# Step 1: Install the Environment
Install the Unsloth environment and dependencies.

In [1]:
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps xformers trl peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-bo62p_vt/unsloth_80e4fe7a4e494f32b163626216031e72
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-bo62p_vt/unsloth_80e4fe7a4e494f32b163626216031e72
  Resolved https://github.com/unslothai/unsloth.git to commit 997f1a7ce5fb7a0319c2b8abe0e7af02e2160efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 128.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 114.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 162.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 111.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 153.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━

In [2]:
!unzip -o v2_agentic_adapters.zip

Archive:  v2_agentic_adapters.zip
   creating: v2_agentic_adapters/
  inflating: v2_agentic_adapters/README.md  
  inflating: v2_agentic_adapters/adapter_model.safetensors  
  inflating: v2_agentic_adapters/adapter_config.json  
  inflating: v2_agentic_adapters/chat_template.jinja  
  inflating: v2_agentic_adapters/tokenizer_config.json  
  inflating: v2_agentic_adapters/tokenizer.json  


# Step 2: Load the SFT Model (CRITICAL)
Load the fine-tuned adapters from our previous step (`v2_agentic_adapters`) using Unsloth. Do NOT load the base model from Hugging Face, and do NOT initialize new PEFT adapters, as we want to continue training our existing SFT weights.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "VKM47/TheraMind-Agentic-V2", # Loads the Phase 2 multi-tasking brain!
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Step 3: Process the Preference Dataset
Load `v3_dpo_preference_data.json`. The `trl` `DPOTrainer` requires standard formatting.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="../data/v3_dpo_preference_data.json", split="train")

def format_dpo(example):
    # DPO requires the prompt, chosen, and rejected fields to be strings
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }
dataset = dataset.map(format_dpo)

# Step 4: Patch Unsloth & Configure DPOTrainer
You MUST patch the `DPOTrainer` for Unsloth before initializing it. Use strict memory and optimization arguments suited for DPO on a 24GB card.

In [3]:
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth handles the reference model implicitly to save VRAM
    tokenizer = tokenizer,
    beta = 0.1, # KL penalty
    train_dataset = dataset,
    max_length = 4096,
    max_prompt_length = 2048,
    args = DPOConfig(
        per_device_train_batch_size = 4,   # Increased to use more VRAM (18-20GB)
        gradient_accumulation_steps = 2,   # Decreased so the model actually updates its weights more often
        warmup_steps = 2,                  # Scaled down to match the new step count
        num_train_epochs = 3,              # Increased to 3 to hit the ~18 step sweet spot
        learning_rate = 5e-6,              # DPO requires a significantly lower LR than SFT
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_torch",
        output_dir = "dpo_medsoap_outputs",
    ),
)
trainer.train()

num_proc must be <= 49. Reducing num_proc to 49 for dataset of size 49.
[datasets.arrow_dataset|WARNING]num_proc must be <= 49. Reducing num_proc to 49 for dataset of size 49.
num_proc must be <= 49. Reducing num_proc to 49 for dataset of size 49.
[datasets.arrow_dataset|WARNING]num_proc must be <= 49. Reducing num_proc to 49 for dataset of size 49.


Applying chat template to train dataset (num_proc=49):   0%|          | 0/49 [00:00<?, ? examples/s]

num_proc must be <= 49. Reducing num_proc to 49 for dataset of size 49.
[datasets.arrow_dataset|WARNING]num_proc must be <= 49. Reducing num_proc to 49 for dataset of size 49.


Tokenizing train dataset (num_proc=49):   0%|          | 0/49 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.000000,17.810413,-6.042602,1.000000,23.853012,-293.832764,-260.300842,0.525378,0.765330,0,0,0
2,0.000000,15.377182,-5.690266,1.000000,21.067448,-265.608826,-252.645111,0.621159,0.640870,No Log,No Log,No Log
3,0.000000,16.361612,-6.041752,1.000000,22.403364,-296.081665,-282.966370,0.381760,0.571516,No Log,No Log,No Log
4,0.000000,15.771438,-4.587753,1.000000,20.359192,-286.701477,-277.705750,0.481969,0.685744,No Log,No Log,No Log
5,0.000000,14.848668,-7.537167,1.000000,22.385834,-282.886627,-280.428955,0.490238,0.634608,No Log,No Log,No Log
6,0.000000,15.288145,-6.190758,1.000000,21.478905,-274.409912,-277.400604,0.487827,0.691950,No Log,No Log,No Log
7,0.000000,18.538120,-4.035498,1.000000,22.573618,-339.560455,-264.916168,0.153335,0.650612,No Log,No Log,No Log
8,0.000000,15.196682,-6.374910,1.000000,21.571590,-268.518768,-275.428192,0.501450,0.630544,No Log,No Log,No Log
9,0.000000,16.583136,-6.002378,1.000000,22.585512,-266.722473,-260.624481,0.495805,0.685457,No Log,No Log,No Log
10,0.000000,15.383933,-6.181357,1.000000,21.565292,-318.398315,-317.760345,0.468768,0.592379,No Log,No Log,No Log


TrainOutput(global_step=21, training_loss=1.858706384770867e-08, metrics={'train_runtime': 66.1245, 'train_samples_per_second': 2.223, 'train_steps_per_second': 0.318, 'total_flos': 0.0, 'train_loss': 1.858706384770867e-08, 'epoch': 3.0})

# Step 5: Save the DPO-Aligned Adapters
Save the final aligned adapters to a new directory.

In [9]:
model.save_pretrained("v3_dpo_aligned_adapters")
tokenizer.save_pretrained("v3_dpo_aligned_adapters")

('v3_dpo_aligned_adapters/tokenizer_config.json',
 'v3_dpo_aligned_adapters/chat_template.jinja',
 'v3_dpo_aligned_adapters/tokenizer.json')

In [10]:
!zip -r v3_dpo_aligned_adapters.zip v3_dpo_aligned_adapters/


  adding: v3_dpo_aligned_adapters/ (stored 0%)
  adding: v3_dpo_aligned_adapters/README.md (deflated 65%)
  adding: v3_dpo_aligned_adapters/adapter_model.safetensors (deflated 8%)
  adding: v3_dpo_aligned_adapters/adapter_config.json (deflated 58%)
  adding: v3_dpo_aligned_adapters/chat_template.jinja (deflated 71%)
  adding: v3_dpo_aligned_adapters/tokenizer_config.json (deflated 45%)
  adding: v3_dpo_aligned_adapters/tokenizer.json (deflated 85%)
